# Syn Bank Share of Wallet Intelligence Engine

Ingestion + profiling only (PLAN.md Section 5, steps 1-3). No modeling logic yet.

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

DATA_DIR = Path.cwd()

print(f"pandas {pd.__version__} | numpy {np.__version__}")
print(f"Data directory: {DATA_DIR.resolve()}")

pandas 3.0.0 | numpy 2.2.1
Data directory: C:\Users\ADMIN\Downloads\Hackathon


## 2. Ingestion

Loads the 3 Syn Bank internal datasets. `transactional_banking.csv` is ~2.8M rows / ~390MB, so this cell may take a minute.

In [2]:
df_txn = pd.read_csv(DATA_DIR / "transactional_banking.csv", parse_dates=["date"])
df_xborder = pd.read_csv(DATA_DIR / "cross_border_payments.csv", parse_dates=["date"])
df_trade = pd.read_csv(DATA_DIR / "trade_finance.csv", parse_dates=["date"])

datasets = {
    "transactional_banking": df_txn,
    "cross_border_payments": df_xborder,
    "trade_finance": df_trade,
}

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} cols")

transactional_banking: 2,802,875 rows x 13 cols
cross_border_payments: 241,117 rows x 13 cols
trade_finance: 20,303 rows x 15 cols


## 3. Profiling

For each dataset: row count, date range, unique entity counts, and null counts per column.

In [3]:
def profile(df: pd.DataFrame, name: str) -> None:
    print("=" * 80)
    print(name)
    print("=" * 80)
    print(f"Rows: {len(df):,}")
    print(f"Date range: {df['date'].min().date()} -> {df['date'].max().date()}")
    print(f"Unique entity_id: {df['entity_id'].nunique()}")
    print(f"Unique entity_name: {df['entity_name'].nunique()}")

    null_summary = pd.DataFrame({
        "null_count": df.isnull().sum(),
        "null_pct": (100 * df.isnull().mean()).round(2),
    })
    print("\nNull counts per column:")
    print(null_summary)
    print()


for name, df in datasets.items():
    profile(df, name)

transactional_banking
Rows: 2,802,875
Date range: 2023-07-01 -> 2026-06-30
Unique entity_id: 20


Unique entity_name: 20



Null counts per column:
                  null_count  null_pct
transaction_id             0      0.00
entity_id                  0      0.00
entity_name                0      0.00
sector                     0      0.00
date                       0      0.00
leg_type                   0      0.00
direction                  0      0.00
amount_zar                 0      0.00
currency                   0      0.00
channel                    0      0.00
beneficiary_name           0      0.00
reference                  0      0.00
memo                 2799218     99.87

cross_border_payments
Rows: 241,117
Date range: 2023-07-01 -> 2026-06-30
Unique entity_id: 20
Unique entity_name: 20



Null counts per column:
                      null_count  null_pct
transaction_id                 0      0.00
entity_id                      0      0.00
entity_name                    0      0.00
sector                         0      0.00
date                           0      0.00
direction                      0      0.00
currency_pair                  0      0.00
value_zar                      0      0.00
counterparty_country        3665      1.52
corridor_type                  0      0.00
beneficiary_name               0      0.00
reference                      0      0.00
memo                      240669     99.81

trade_finance
Rows: 20,303
Date range: 2023-07-01 -> 2026-06-30
Unique entity_id: 20
Unique entity_name: 20

Null counts per column:
                            null_count  null_pct
instrument_id                        0      0.00
entity_id                            0      0.00
entity_name                          0      0.00
sector                               0     

## 4. Entity list sanity check

Confirms the entities present in the data match the 20 names confirmed in PLAN.md Section 3.

In [4]:
PLAN_ENTITIES = {
    "Anglo American", "AngloGold Ashanti", "Aspen Pharmacare", "BHP Group",
    "Bid Corporation", "Clicks Group", "Glencore", "Gold Fields", "MTN Group",
    "NEPI Rockcastle", "Naspers", "OUTsurance Group", "Pepkor Holdings",
    "Prosus", "Sanlam", "Shaftesbury Capital plc", "Shoprite Holdings",
    "The Bidvest Group", "Valterra Platinum", "Vodacom Group",
}

data_entities = set()
for name, df in datasets.items():
    data_entities |= set(df["entity_name"].unique())

print(f"Entities found across all 3 datasets: {len(data_entities)}")
print(f"Entities listed in PLAN.md Section 3: {len(PLAN_ENTITIES)}")

missing_from_data = PLAN_ENTITIES - data_entities
extra_in_data = data_entities - PLAN_ENTITIES

if not missing_from_data and not extra_in_data:
    print("\nMATCH -- data entity list is identical to PLAN.md Section 3.")
else:
    print("\nMISMATCH")
    if missing_from_data:
        print(f"  In PLAN.md but not in data: {sorted(missing_from_data)}")
    if extra_in_data:
        print(f"  In data but not in PLAN.md: {sorted(extra_in_data)}")

for name, df in datasets.items():
    entities_here = set(df["entity_name"].unique())
    if entities_here != data_entities:
        print(f"NOTE: {name} does not cover all entities -- missing {data_entities - entities_here}")

Entities found across all 3 datasets: 20
Entities listed in PLAN.md Section 3: 20

MATCH -- data entity list is identical to PLAN.md Section 3.
